# Skeletons

**Geometry type:** `skeleton`

Tree-structured graphs aligned to the SWC morphology convention. Each
node has a position, radius, SWC compartment type, and one parent.

1. Generate a synthetic neuron morphology
2. Write as a skeleton store
3. Open lazily and inspect SWC attributes
4. Read the full morphology
5. Multi-skeleton store (connectome use case)
6. Spatial bounding-box query
7. SWC round-trip (ingest + export)
8. Validate


In [ ]:
import numpy as np, tempfile, os
from zarr_vectors.lazy import ZarrVectorStore
from zarr_vectors.types.graphs import write_graph, read_graph
from zarr_vectors.validate import validate

_tmpdir = tempfile.mkdtemp(prefix="zvf_examples_")
STORE      = os.path.join(_tmpdir, "neuron.zarrvectors")
STORE_MULTI = os.path.join(_tmpdir, "connectome.zarrvectors")
SWC_IN     = os.path.join(_tmpdir, "neuron.swc")
SWC_OUT    = os.path.join(_tmpdir, "neuron_roundtrip.swc")
print("Stores:", STORE, STORE_MULTI)


## 1 · Generate a synthetic neuron morphology

In [ ]:
rng = np.random.default_rng(3)

def generate_tree(n_nodes, origin, rng):
    """Generate a random branching tree with SWC attributes."""
    positions = np.zeros((n_nodes, 3), dtype=np.float32)
    positions[0] = origin                      # soma at origin
    parent_idx = np.full(n_nodes, -1, dtype=np.int32)

    for i in range(1, n_nodes):
        # Attach to a random existing node (creates branching)
        parent_idx[i] = rng.integers(0, i)
        step = rng.normal(0, 4, 3).astype(np.float32)
        positions[i] = positions[parent_idx[i]] + step

    # SWC compartment types: 0=soma, 1=axon, 2=basal dendrite
    swc_type = np.ones(n_nodes, dtype=np.int32) * 3   # basal dendrite default
    swc_type[0] = 1    # soma
    swc_type[1:n_nodes//5] = 2   # axon: first ~20% of nodes

    radii = np.where(swc_type == 1,
                     rng.uniform(8, 15, n_nodes),     # soma: large radius
                     rng.uniform(0.3, 2.0, n_nodes)   # processes: small
                    ).astype(np.float32)

    # Edges: [child, parent] pairs (root has parent = -1, stored separately)
    edges = np.column_stack([
        np.arange(1, n_nodes, dtype=np.int32),
        parent_idx[1:],
    ])
    return positions, edges, swc_type, radii

n_nodes = 1_200
positions, edges, swc_type, radii = generate_tree(
    n_nodes, origin=np.array([500., 500., 500.], dtype=np.float32), rng=rng
)

print(f"Generated neuron with {n_nodes} nodes and {len(edges)} edges")
print(f"SWC types: { dict(zip(*np.unique(swc_type, return_counts=True))) }")
print(f"Radius range: [{radii.min():.2f}, {radii.max():.2f}] µm")


## 2 · Write skeleton store

In [ ]:
write_graph(
    STORE,
    positions=positions,
    edges=edges,             # [child, parent] pairs; root has parent -1 internally
    chunk_shape=(200.0, 200.0, 200.0),
    bin_shape=(50.0, 50.0, 50.0),
    geometry_type="skeleton",
    is_tree=True,
    vertex_attributes={
        "radius":   radii,
        "swc_type": swc_type,
        "swc_id":   np.arange(n_nodes, dtype=np.int64),  # original node IDs
    },
    coordinate_system="RAS",
    axis_units="micrometer",
)
print("Write complete.")


## 3 · Open lazily and inspect SWC attributes

In [ ]:
store = ZarrVectorStore(STORE)
print(f"geometry_type : {store.geometry_type}")
print(f"n_objects     : {store.n_objects}  (1 skeleton)")
print(f"vertex_count  : {store.vertex_count(0):,}")
print(f"bounding_box  : lo={store.bounding_box[0].round(1)}"
      f"  hi={store.bounding_box[1].round(1)}")


## 4 · Read the full morphology

In [ ]:
result = read_graph(STORE)
print(f"node_count    : {result['node_count']:,}")
print(f"edge_count    : {result['edge_count']:,}")
print(f"positions     : {result['positions'].shape}")
print(f"radius range  : [{result['attributes']['radius'].min():.2f}, "
      f"{result['attributes']['radius'].max():.2f}] µm")

# Compartment type breakdown
types, counts = np.unique(result['attributes']['swc_type'], return_counts=True)
print("\nCompartment types:")
type_names = {1: "soma", 2: "axon", 3: "basal dendrite", 4: "apical dendrite"}
for t, c in zip(types, counts):
    print(f"  type {t} ({type_names.get(int(t), 'other'):>15s}): {c:>5} nodes")


## 5 · Multi-skeleton store (connectome)

In [ ]:
# Generate 50 small neurons spread across a larger volume
n_neurons = 50
all_pos, all_edges_list, all_types, all_radii = [], [], [], []

for i in range(n_neurons):
    origin = rng.uniform(0, 2000, 3).astype(np.float32)
    p, e, t, r = generate_tree(rng.integers(200, 600), origin, rng)
    all_pos.append(p)
    all_edges_list.append(e)
    all_types.append(t)
    all_radii.append(r)

# Write as multi-skeleton store — one object per neuron
import numpy as np
write_graph(
    STORE_MULTI,
    positions=all_pos,           # list of per-neuron position arrays
    edges=all_edges_list,        # list of per-neuron edge arrays
    chunk_shape=(500., 500., 500.),
    bin_shape=(100., 100., 100.),
    geometry_type="skeleton",
    is_tree=True,
    vertex_attributes={
        "radius":   all_radii,
        "swc_type": all_types,
    },
)
print(f"Multi-skeleton store: {n_neurons} neurons")

store_m = ZarrVectorStore(STORE_MULTI)
print(f"n_objects     : {store_m.n_objects}")
print(f"vertex_count  : {store_m.vertex_count(0):,}")


## 5b · Read a single neuron from the connectome store

In [ ]:
# Read neuron 7 by object ID
result_7 = read_graph(STORE_MULTI, object_ids=[7])
print(f"Neuron 7 has {result_7['node_count']} nodes and {result_7['edge_count']} edges")

# Spatial query: which neurons have processes in a specific region?
lo_q = np.array([800., 800., 800.])
hi_q = np.array([1200., 1200., 1200.])
bbox_res = read_graph(STORE_MULTI, bbox=(lo_q, hi_q), return_object_ids=True)
print(f"\nNeurons with nodes in [{lo_q}] – [{hi_q}]:")
print(f"  {bbox_res['object_ids']}")


## 6 · SWC round-trip

In [ ]:
# Write a minimal synthetic SWC
with open(SWC_IN, "w") as f:
    f.write("# Generated by zarr-vectors example\n")
    for i in range(n_nodes):
        p_id = -1 if i == 0 else int(edges[i-1, 1]) + 1
        f.write(f"{i+1} {swc_type[i]} "
                f"{positions[i,0]:.3f} {positions[i,1]:.3f} {positions[i,2]:.3f} "
                f"{radii[i]:.3f} {p_id}\n")
print(f"Wrote {n_nodes}-node SWC to {SWC_IN}")

# Ingest SWC → ZVF
STORE_SWC = os.path.join(_tmpdir, "from_swc.zarrvectors")
from zarr_vectors.ingest.swc import ingest_swc
ingest_swc(SWC_IN, STORE_SWC, chunk_shape=(200., 200., 200.))
r_swc = read_graph(STORE_SWC)
print(f"Ingested: {r_swc['node_count']} nodes, {r_swc['edge_count']} edges")

# Export back to SWC
from zarr_vectors.export.swc import export_swc
export_swc(STORE_SWC, SWC_OUT)
print(f"Exported SWC to {SWC_OUT}")

# Verify round-trip
lines_in  = [l for l in open(SWC_IN)  if not l.startswith("#")]
lines_out = [l for l in open(SWC_OUT) if not l.startswith("#")]
print(f"\nInput  rows: {len(lines_in)}")
print(f"Output rows: {len(lines_out)}")


## 7 · Validate

In [ ]:
rv = validate(STORE, level=4)   # level 4 checks tree topology
print(rv.summary())


## Summary

| Step | API |
|------|-----|
| Write | `write_graph(path, positions, edges, geometry_type="skeleton", is_tree=True, vertex_attributes=...)` |
| Read | `read_graph(path)` |
| Read by ID | `read_graph(path, object_ids=[k])` |
| Bbox query | `read_graph(path, bbox=(lo, hi))` |
| SWC ingest | `ingest_swc(swc_path, store_path)` |
| SWC export | `export_swc(store_path, swc_path)` |
